# 01 — Training GALERIA — MODEL v3 (CLIP frozen + linear probe)

## MODEL v3 (CLIP ViT-B-32, frozen + classifier head) -- siap di-run ulang kapan saja

Berbeda dari v1/v2 (`convnext_small` fine-tuned penuh): di sini backbone CLIP
**dibekukan total** (TIDAK ada gradient ke visual encoder) -- hanya classifier
head kecil (`Linear` atau 1-hidden MLP, lihat `configs/config.yaml` ->
`baseline_clip.classifier`) yang dilatih, di ATAS embedding yang sudah
diekstrak sekali di awal. Ini jauh lebih murah dari fine-tuning penuh (menit,
bukan jam) -- itulah nilai jual utama CLIP sbg pembanding v3.

**Dataset SAMA PERSIS dgn v2** (`data/raw/*_split.csv`, 33.937 gambar, 22
kelas) -- supaya perbandingan v2 vs v3 murni soal "fine-tune penuh vs frozen
+ linear probe", bukan tercampur beda data.

**Tidak menyentuh** checkpoint/output v1 maupun v2 -- semua artefak run ini
ditulis ke `checkpoints/v3_clip/`, `outputs/v3_clip/`, `export/v3_clip/`,
`data/cache/v3_clip/`.

**Prasyarat:** `pip install open-clip-torch` (lihat `requirements.txt`,
sudah ditambahkan tapi belum ter-install di environment sampai dijalankan
manual). Download bobot pretrained CLIP terjadi otomatis saat pertama kali
`open_clip.create_model_and_transforms(...)` dipanggil (~600 MB, butuh
internet sekali saja -- disimpan ke cache torch/open_clip lokal).

## 1. Setup

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

# pindah ke folder ml-visual-search/ supaya `import src` & path config jalan
_p = Path.cwd()
while not ((_p / "src").is_dir() and (_p / "configs").is_dir()):
    if _p == _p.parent:
        raise RuntimeError("folder ml-visual-search/ (berisi src/ & configs/) tidak ketemu")
    _p = _p.parent
os.chdir(_p)
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

import json, time
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt

try:
    import open_clip
except ImportError as e:
    raise ImportError(
        "open_clip belum terinstall -- jalankan `pip install open-clip-torch` "
        "dulu (lihat requirements.txt)."
    ) from e

print("working dir :", os.getcwd())
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("open_clip   :", open_clip.__version__)

In [ ]:
from src.utils import load_config, set_seed, get_device, resolve_path
from src.dataset import build_label_mapping, WikiArtDataset
from src.evaluate import compute_metrics

cfg = load_config("configs/config.yaml")
clip_cfg = cfg["baseline_clip"]
assert clip_cfg["enabled"], "baseline_clip.enabled harus true di config.yaml"

CKPT_DIR = resolve_path(clip_cfg["checkpoint_dir"]);  CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR  = resolve_path(clip_cfg["output_dir"]);        OUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(cfg["seed"])
device = get_device(cfg["device"])
print("device:", device)
print(json.dumps(clip_cfg, indent=2))

## 2. Muat CLIP pretrained (frozen) + label mapping

`label_to_idx` dibangun dari `data.train_csv` yang SAMA dgn v2 (22 kelas) --
dipanggil lewat `build_label_mapping(cfg)`, bukan di-hardcode, supaya tetap
konsisten kalau dataset berubah lagi di masa depan.

In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    clip_cfg["model_name"], pretrained=clip_cfg["pretrained"]
)
clip_model = clip_model.to(device).eval()
for p in clip_model.parameters():
    p.requires_grad_(False)   # BEKUKAN total -- tidak ada fine-tuning backbone

label_to_idx = build_label_mapping(cfg)
idx_to_label = {v: k for k, v in label_to_idx.items()}
num_classes  = len(label_to_idx)
class_names  = [idx_to_label[i] for i in range(num_classes)]
print(f"{num_classes} kelas:", class_names)
print("CLIP visual encoder dibekukan (requires_grad=False utk semua parameter).")

## 3. Ekstrak embedding CLIP (SEKALI, tanpa augmentasi, di-cache ke disk)

Karena backbone dibekukan, augmentasi di training set tidak melatih apa pun
di backbone -- hanya berguna kalau classifier head di-train langsung di atas
gambar mentah tiap epoch (mahal). Di sini kita ekstrak embedding
DETERMINISTIK (pakai `clip_preprocess` yang sama utk train/val/test) satu
kali, lalu classifier head dilatih di atas embedding yang sudah di-cache --
jauh lebih cepat, trade-off yang wajar utk linear probe (catat di laporan).

**Di-cache ke `data/cache/v3_clip/split_embeddings/` (`.npz` per split)** --
kalau sel ini dijalankan ulang (mis. setelah tuning classifier head di section
4-5) langsung load dari cache, TIDAK ekstrak ulang dari gambar (itu yang
paling lama, beberapa menit). Hapus folder cache itu manual kalau mau
paksa ekstrak ulang (mis. ganti `clip_cfg.model_name`).

In [ ]:
@torch.no_grad()
def extract_clip_embeddings(csv_key: str, split_name: str, desc: str):
    # subfolder per model CLIP -- ganti model_name/pretrained di config tidak akan
    # diam-diam memakai cache dari model lain
    model_tag = f"{clip_cfg['model_name']}__{clip_cfg['pretrained']}"
    cache_path = resolve_path(clip_cfg["embedding_cache_dir"]) / model_tag / f"{split_name}.npz"
    if cache_path.exists():
        cached = np.load(cache_path)
        print(f"{desc}: load dari cache -> {cache_path}")
        return cached["embeddings"], cached["labels"]

    ds = WikiArtDataset(
        csv_path=cfg["data"][csv_key],
        image_dir=cfg["data"]["image_dir"],
        label_column=cfg["data"]["label_column"],
        label_to_idx=label_to_idx,
        transform=clip_preprocess,
        filename_column=cfg["data"]["filename_column"],
    )
    loader = torch.utils.data.DataLoader(
        ds, batch_size=clip_cfg["batch_size"], shuffle=False,
        num_workers=cfg["train"]["num_workers"], pin_memory=True,
    )
    from tqdm import tqdm
    embs, labels = [], []
    for images, lbl in tqdm(loader, desc=desc, leave=False):
        images = images.to(device, non_blocking=True)
        feat = clip_model.encode_image(images)
        feat = torch.nn.functional.normalize(feat, dim=1)
        embs.append(feat.cpu().numpy())
        labels.append(lbl.numpy())
    embeddings = np.concatenate(embs, axis=0)
    labels = np.concatenate(labels, axis=0)

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cache_path, embeddings=embeddings, labels=labels)
    return embeddings, labels


t0 = time.time()
train_emb, train_lbl = extract_clip_embeddings("train_csv", "train", "embed train")
val_emb,   val_lbl   = extract_clip_embeddings("val_csv",   "val",   "embed val")
test_emb,  test_lbl  = extract_clip_embeddings("test_csv",  "test",  "embed test")
print(f"shapes: train {train_emb.shape} val {val_emb.shape} test {test_emb.shape}")
print(f"ekstraksi embedding (atau load cache): {(time.time()-t0)/60:.1f} menit")

## 4. Classifier head

`hidden_dim: 0` di config -> Linear murni (linear probe klasik). `hidden_dim > 0`
-> 1-hidden-layer MLP (lebih ekspresif, masih jauh lebih kecil dari fine-tuning
backbone, dan backbone CLIP tetap beku).

**Standardisasi fitur** (`standardize: true`): embedding CLIP sudah L2-norm,
akibatnya tiap dimensi nilainya sangat kecil (std ~0.03) -> head belajar
lambat & butuh bobot besar (yang justru dilawan weight decay). Fitur dinormalisasi
ke mean 0 / std 1 per dimensi pakai statistik **train saja** (val/test pakai
statistik train yg sama -- tidak bocor). Statistik ini disimpan di checkpoint
supaya `03_demo_inference.ipynb` memakai normalisasi yang identik.

Setting saat ini hasil sweep 25 konfigurasi (lihat komentar `configs/config.yaml`
-> `baseline_clip.classifier`): val macro-F1 0.667 vs 0.629 run sebelumnya.

In [ ]:
clf_cfg = clip_cfg["classifier"]
embed_dim = train_emb.shape[1]

if clf_cfg.get("standardize", False):
    feat_mean = train_emb.mean(0).astype(np.float32)
    feat_std  = (train_emb.std(0) + 1e-6).astype(np.float32)
else:
    feat_mean = np.zeros(embed_dim, dtype=np.float32)
    feat_std  = np.ones(embed_dim, dtype=np.float32)

def prep(x):
    return (x - feat_mean) / feat_std

print(f"std per dimensi: mentah {train_emb.std(0).mean():.4f} -> setelah prep {prep(train_emb).std(0).mean():.4f}")


def build_classifier(cfg_c, in_dim, n_cls):
    if cfg_c["hidden_dim"] and cfg_c["hidden_dim"] > 0:
        return nn.Sequential(
            nn.Dropout(cfg_c.get("input_dropout", 0.0)),
            nn.Linear(in_dim, cfg_c["hidden_dim"]), nn.GELU(),
            nn.Dropout(cfg_c["dropout"]),
            nn.Linear(cfg_c["hidden_dim"], n_cls),
        )
    return nn.Sequential(nn.Dropout(cfg_c.get("input_dropout", 0.0)), nn.Linear(in_dim, n_cls))


set_seed(cfg["seed"])   # reset RNG tepat sebelum init -> hasil bisa direproduksi
classifier = build_classifier(clf_cfg, embed_dim, num_classes).to(device)

n_params = sum(p.numel() for p in classifier.parameters())
print(classifier)
print(f"classifier head: {n_params:,} parameter (vs convnext_small ~31M trainable di v1/v2)")

## 5. Training classifier head (di atas embedding yang sudah di-cache)

**Mitigasi imbalance**: training set 44x imbalance (Impressionism 5269 vs
Pointillism 120). Tanpa penanganan, classifier condong ke kelas mayoritas --
gejala run pertama: accuracy 0.625 lumayan tapi macro-F1 cuma 0.517. Dipakai
**class-weighted `CrossEntropyLoss`** (bobot = inverse frekuensi kelas di train,
dinormalisasi). v1/v2 pakai `WeightedRandomSampler` (mekanisme beda, tujuan sama:
paksa model "peduli" ke kelas minoritas).

**Regularisasi & jadwal LR** (hasil sweep, lihat config): dropout di input +
hidden, `weight_decay` 0.05, cosine LR schedule, early stopping pada val
macro-F1. Bobot yang disimpan = epoch dgn val macro-F1 TERBAIK, bukan epoch
terakhir -- jadi kalau di akhir training mulai overfit, checkpoint tetap aman.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

# embedding sudah kecil (puluhan ribu x 512) -> taruh semua di device sekaligus,
# batching manual (jauh lebih cepat dari DataLoader utk data sekecil ini)
Xtr = torch.tensor(prep(train_emb), dtype=torch.float32, device=device)
Ytr = torch.tensor(train_lbl, dtype=torch.long, device=device)
Xva = torch.tensor(prep(val_emb), dtype=torch.float32, device=device)
Yva = torch.tensor(val_lbl, dtype=torch.long, device=device)
Xte = torch.tensor(prep(test_emb), dtype=torch.float32, device=device)

weight_tensor = None
if clf_cfg.get("use_class_weights", False):
    class_count = np.bincount(train_lbl, minlength=num_classes).clip(min=1)
    class_weight = (1.0 / class_count)
    class_weight = class_weight / class_weight.sum() * num_classes   # rata-rata bobot = 1
    weight_tensor = torch.tensor(class_weight, dtype=torch.float32, device=device)
    print("class-weighted CrossEntropyLoss aktif -- bobot per kelas (min..max):",
          f"{class_weight.min():.2f}..{class_weight.max():.2f}")
criterion = nn.CrossEntropyLoss(weight=weight_tensor,
                                label_smoothing=clf_cfg.get("label_smoothing", 0.0))

optimizer = torch.optim.AdamW(classifier.parameters(),
                              lr=clf_cfg["learning_rate"], weight_decay=clf_cfg["weight_decay"])
scheduler = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=clf_cfg["epochs"])
             if clf_cfg.get("scheduler", "none") == "cosine" else None)


@torch.no_grad()
def predict(X, bs=4096):
    classifier.eval()
    return torch.cat([classifier(X[i:i + bs]) for i in range(0, len(X), bs)])


history = {"train_loss": [], "val_loss": [], "train_macro_f1": [], "val_macro_f1": [],
           "val_accuracy": [], "lr": []}
step_losses = []   # loss TIAP BATCH -- bukti gradient descent, lihat section 5b
best_f1, best_state, no_improve = -1.0, None, 0
patience = clf_cfg.get("early_stopping_patience", 20)
bs = clf_cfg["batch_size"]
n = len(Xtr)
t0 = time.time()

for epoch in range(1, clf_cfg["epochs"] + 1):
    classifier.train()
    tr_loss = 0.0
    perm = torch.randperm(n, device=device)
    for i in range(0, n, bs):
        idx = perm[i:i + bs]
        optimizer.zero_grad()
        loss = criterion(classifier(Xtr[idx]), Ytr[idx])
        loss.backward(); optimizer.step()
        step_losses.append(loss.item())
        tr_loss += loss.item() * len(idx)
    tr_loss /= n
    lr_now = optimizer.param_groups[0]["lr"]
    if scheduler is not None:
        scheduler.step()

    tr_logits = predict(Xtr)
    tr_f1 = f1_score(train_lbl, tr_logits.argmax(1).cpu().numpy(), average="macro")
    va_logits = predict(Xva)
    va_loss = criterion(va_logits, Yva).item()
    va_pred = va_logits.argmax(1).cpu().numpy()
    va_f1 = f1_score(val_lbl, va_pred, average="macro")
    va_acc = accuracy_score(val_lbl, va_pred)

    for k, v in (("train_loss", tr_loss), ("val_loss", va_loss), ("train_macro_f1", tr_f1),
                 ("val_macro_f1", va_f1), ("val_accuracy", va_acc), ("lr", lr_now)):
        history[k].append(v)
    print(f"epoch {epoch:3d}/{clf_cfg['epochs']} | lr {lr_now:.1e} | train loss {tr_loss:.3f} | "
          f"val loss {va_loss:.3f} | train f1 {tr_f1:.3f} val f1 {va_f1:.3f} "
          f"(gap {tr_f1 - va_f1:+.3f}) acc {va_acc:.3f}")

    if va_f1 > best_f1:
        best_f1, best_epoch, no_improve = va_f1, epoch, 0
        best_state = {k: v.detach().cpu().clone() for k, v in classifier.state_dict().items()}
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping -- val macro-F1 tidak membaik {patience} epoch."); break

classifier.load_state_dict(best_state)
print(f"\nbest val macro-F1 = {best_f1:.4f} (epoch {best_epoch}) | total {(time.time()-t0)/60:.1f} menit")

resolve_path(OUT_DIR / "history.json").write_text(
    json.dumps({**history, "step_losses": step_losses}, indent=2), encoding="utf-8")

## 5b. Visualisasi Training — bukti gradient descent & diagnosis overfitting

1. **Loss per step** -- turun = gradient descent benar-benar jalan (bukan
   stuck). Kalau kurva ini flat dari awal, BARU itu tanda ada yang salah
   (lr=0, gradien tidak sampai ke classifier, dll).
2. **Learning curve train vs val** (loss & macro-F1) -- ini yang jawab
   "kenapa hasilnya jelek": kalau `train_loss` terus turun tapi `val_loss`
   balik naik (atau gap `train_macro_f1 - val_macro_f1` melebar), itu
   **overfitting** -- classifier menghafal embedding train, BUKAN gradient
   descent gagal. Overfitting dan "belum konvergen" butuh obat yang
   berlawanan (early stop vs epoch lebih banyak), makanya penting dibedakan
   lewat plot ini, bukan cuma dari angka akhir.

In [ ]:
plt.figure(figsize=(13, 4))
plt.plot(step_losses, lw=0.5, alpha=0.35, color="gray", label="loss per batch")
k = max(1, len(step_losses) // 60)
if len(step_losses) > k:
    ma = np.convolve(step_losses, np.ones(k) / k, mode="valid")
    plt.plot(range(k - 1, len(step_losses)), ma, lw=2, color="tab:red", label=f"moving avg ({k})")
plt.xlabel("training step (batch)"); plt.ylabel("CrossEntropy loss")
plt.title(f"v3 CLIP: loss turun sepanjang {len(step_losses)} step -> gradient descent bekerja")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(resolve_path(OUT_DIR / "train_step_loss.png"), dpi=110); plt.show()
print(f"loss step 1 : {step_losses[0]:.3f}  ->  step {len(step_losses)} : {step_losses[-1]:.3f}"
      f"  (turun {step_losses[0]-step_losses[-1]:.3f})")

In [ ]:
ep = range(1, len(history["train_loss"]) + 1)
best_epoch_plot = int(np.argmax(history["val_macro_f1"])) + 1
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(ep, history["train_loss"], "o-", ms=2, label="train")
ax[0].plot(ep, history["val_loss"],  "s-", ms=2, label="val")
ax[0].axvline(best_epoch_plot, color="gray", ls="--", lw=1, label=f"epoch terbaik ({best_epoch_plot})")
ax[0].set_title("Loss (CrossEntropy)"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(ep, history["train_macro_f1"], "o-", ms=2, label="train")
ax[1].plot(ep, history["val_macro_f1"],  "s-", ms=2, label="val")
ax[1].axvline(best_epoch_plot, color="gray", ls="--", lw=1, label="bobot yang disimpan")
ax[1].set_title("macro-F1"); ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.suptitle("v3 CLIP: learning curve -- train (bulat) vs val (kotak)", fontsize=12)
plt.tight_layout(); plt.savefig(resolve_path(OUT_DIR / "train_learning_curve.png"), dpi=110); plt.show()

i = best_epoch_plot - 1
gap_best = history["train_macro_f1"][i] - history["val_macro_f1"][i]
print(f"epoch terbaik (bobot yang DISIMPAN): {best_epoch_plot}/{len(ep)}")
print(f"  train macro-F1 {history['train_macro_f1'][i]:.3f} | val macro-F1 {history['val_macro_f1'][i]:.3f}"
      f" | gap {gap_best:+.3f}")
if gap_best > 0.15:
    print("\nGap train-val besar, TAPI val macro-F1 masih naik sampai epoch terbaik -> head")
    print("bisa memisahkan data train hampir sempurna di ruang embedding CLIP, namun pola itu")
    print("tidak sepenuhnya generalize. Sweep 25 konfigurasi (dropout s/d 0.5, weight_decay s/d")
    print("0.05, label smoothing, Linear vs MLP) mentok di val macro-F1 ~0.66 -> ini batas")
    print("kualitas embedding CLIP ViT-B-32 yang DIBEKUKAN utk membedakan gaya lukisan, bukan")
    print("masalah gradient descent / setting head. Lever berikutnya: backbone CLIP lebih besar.")
else:
    print("\nGap train-val macro-F1 wajar -- bukan overfit parah.")

## 6. Evaluasi test set

In [ ]:
te_logits = predict(Xte)
y_prob = torch.softmax(te_logits, dim=1).cpu().numpy()
y_pred = y_prob.argmax(1)
y_true = test_lbl

metrics = compute_metrics(y_true, y_pred, y_prob, idx_to_label,
                          subset_min_support=cfg.get("eval", {}).get("subset_min_support", 8))

print(f"TEST (n={metrics['n_test']})")
for k, lab in [("accuracy", "accuracy      "), ("top3_accuracy", "top-3 accuracy"),
               ("macro_f1", "macro-F1      "), ("weighted_f1", "weighted-F1   "),
               ("mse", "MSE / Brier   ")]:
    print(f"  {lab}: {metrics[k]:.3f}")

print("\nkonteks (lihat notebooks/04_compare_models.ipynb utk tabel lengkap):")
print("  - v1 (convnext_small fine-tuned, 11 kelas, 5.659 gambar) : test macro-F1 0.78")
print("  - v2 (convnext_small fine-tuned, 22 kelas, 33.937 gambar): val macro-F1 0.67 (belum final)")
print(f"  - v3 (CLIP frozen + head, 22 kelas, 33.937 gambar)        : test macro-F1 {metrics['macro_f1']:.2f}")
print("  - CATATAN: v3 TIDAK fine-tune backbone sama sekali -- yang dilatih cuma head kecil.")

metrics["model_version"] = "v3_clip"
metrics["clip_model_name"] = clip_cfg["model_name"]
metrics["clip_pretrained"] = clip_cfg["pretrained"]
metrics["classifier_config"] = clf_cfg
metrics["best_val_macro_f1"] = best_f1
metrics["best_epoch"] = best_epoch
resolve_path(OUT_DIR / "metrics_report.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")

## 7. Confusion matrix + simpan checkpoint

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(10, 9))
try:
    import seaborn as sns
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", annot_kws={"size": 7},
                xticklabels=class_names, yticklabels=class_names, ax=ax, cbar=True)
except ImportError:
    im = ax.imshow(cm_norm, cmap="Blues")
    ax.set_xticks(range(num_classes), class_names, rotation=90)
    ax.set_yticks(range(num_classes), class_names); fig.colorbar(im, ax=ax)
ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual"); ax.set_title("Confusion matrix (test) -- v3 CLIP")
plt.tight_layout(); plt.savefig(resolve_path(OUT_DIR / "confusion_matrix.png"), dpi=110); plt.show()

torch.save({
    "classifier_state_dict": classifier.state_dict(),
    "label_to_idx": label_to_idx,
    "clip_model_name": clip_cfg["model_name"],
    "clip_pretrained": clip_cfg["pretrained"],
    "classifier_config": clf_cfg,
    "embed_dim": embed_dim,
    # WAJIB ikut disimpan -- inference (03_demo) harus pakai normalisasi yang sama persis
    "feat_mean": feat_mean,
    "feat_std": feat_std,
    "val_metrics": {"macro_f1": best_f1, "epoch": best_epoch},
}, CKPT_DIR / "best.pth")
print("checkpoint tersimpan ->", CKPT_DIR / "best.pth")
print("(berisi classifier head + statistik standardisasi -- bobot CLIP backbone tidak diubah,")
print(" di-load ulang dari cache open_clip tiap kali notebook dijalankan.)")

## 8. Catatan interpretasi (untuk laporan)

- **v3 BUKAN fine-tuning** -- CLIP ViT-B-32 (pretrained `laion2b_s34b_b79k`,
  bukan ImageNet) dibekukan total, hanya classifier head kecil (~0,55 jt
  parameter) yang dilatih di atas embedding yang sudah diekstrak.
- **Riwayat tuning head** (val macro-F1, dataset & embedding sama persis):

  | Run | Perubahan | val macro-F1 |
  |---|---|---|
  | 1 | Linear, 30 epoch, tanpa penanganan imbalance | 0.517 |
  | 2 | + class-weighted loss, 150 epoch | 0.614 |
  | 3 | MLP 256, dropout 0.2, wd 1e-4 (overfit parah) | 0.629 |
  | 4 | + standardisasi fitur, MLP 1024, dropout input 0.25 / hidden 0.5, wd 0.05, cosine LR (hasil sweep 25 konfigurasi) | **0.667** (test 0.666, akurasi 0.711) |

- **Batas yang ditemukan**: banyak konfigurasi head berbeda mentok di ~0.66,
  dan gap train-val macro-F1 tetap ~0.3 walau regularisasi dinaikkan --
  artinya plafonnya ada di embedding CLIP yang dibekukan, bukan di head.
  Kelas yang paling sering tertukar memang aliran yang overlap secara
  historis (Fauvism~Post_Impressionism, Rococo~Baroque, Pointillism~
  Impressionism, sub-periode Renaissance). Ini argumen konkret kenapa
  fine-tuning backbone (v1/v2) tetap relevan untuk domain spesifik seperti ini.
- **Dataset SAMA dgn v2** (22 kelas, 33.937 gambar, split sama) -- perbandingan
  v2 vs v3 di `notebooks/04_compare_models.ipynb` murni soal strategi training.
- **Biaya**: ekstraksi embedding sekali (beberapa menit, lalu di-cache);
  training head ~30 detik (vs ~6 jam utk v2 fine-tuning penuh 40 epoch).
- Artefak: `outputs/v3_clip/history.json`, `metrics_report.json`,
  `train_step_loss.png`, `train_learning_curve.png`, `confusion_matrix.png`;
  `checkpoints/v3_clip/best.pth` (classifier head + statistik standardisasi).